# SONYC Dataset Exploration
CLAP embeddings + UMAP + clustering. Run cells top to bottom.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import umap
import warnings
warnings.filterwarnings('ignore')

# load embeddings
embs = np.load('outputs/embeddings.npy')          # (18510, 512)
embs = normalize(embs, norm='l2')         # unit sphere (should already be, just in case)

# load metadata
meta = []
with open('outputs/embeddings_meta.jsonl') as f:
    for line in f:
        meta.append(json.loads(line))

print(f'embeddings: {embs.shape}')
print(f'metadata rows: {len(meta)}')
print(f'sample: {meta[0]}')

In [3]:
# coarse class label for each clip (from first labeled class, or 'unlabeled')
# fine class -> coarse class mapping
FINE_TO_COARSE = {
    'small-sounding-engine': 'engine', 'medium-sounding-engine': 'engine', 'large-sounding-engine': 'engine',
    'rock-drill': 'machinery', 'jackhammer': 'machinery', 'hoe-ram': 'machinery', 'pile-driver': 'machinery',
    'non-machinery-impact': 'impact',
    'chainsaw': 'saw', 'small-medium-rotating-saw': 'saw', 'large-rotating-saw': 'saw',
    'car-horn': 'alert', 'car-alarm': 'alert', 'siren': 'alert', 'reverse-beeper': 'alert',
    'stationary-music': 'music', 'mobile-music': 'music', 'ice-cream-truck': 'music',
    'person-or-small-group-talking': 'voice', 'person-or-small-group-shouting': 'voice',
    'large-crowd': 'voice', 'amplified-speech': 'voice',
    'dog-barking-whining': 'dog',
}
COARSE_COLORS = {
    'engine': '#ff6b6b', 'machinery': '#ff9f43', 'impact': '#ffd32a',
    'saw': '#26de81', 'alert': '#fd79a8', 'music': '#a29bfe',
    'voice': '#74b9ff', 'dog': '#55efc4', 'unlabeled': '#333344',
}

def get_coarse(m):
    for c in m.get('classes', []):
        # match fine class name to coarse
        for fine, coarse in FINE_TO_COARSE.items():
            if fine in c:
                return coarse
    return 'unlabeled'

labels = [get_coarse(m) for m in meta]
colors = [COARSE_COLORS[l] for l in labels]
boroughs = [m.get('borough', '') for m in meta]
hours = [m.get('hour', -1) for m in meta]

from collections import Counter
print('label distribution:', Counter(labels).most_common())

label distribution: [('unlabeled', 17276), ('engine', 607), ('voice', 244), ('impact', 182), ('alert', 115), ('music', 36), ('machinery', 22), ('saw', 17), ('dog', 11)]


In [ ]:
# UMAP — takes ~3-5 min on 18k clips
# reduce first with PCA for speed
from sklearn.decomposition import PCA

print('PCA to 50 dims...')
pca = PCA(n_components=50, random_state=42)
embs_pca = pca.fit_transform(embs)
print(f'  variance explained: {pca.explained_variance_ratio_.sum():.1%}')

print('UMAP to 2D...')
reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric='cosine', random_state=13)
embs_2d = reducer.fit_transform(embs_pca)
print('done.')

PCA to 50 dims...
  variance explained: 94.5%
UMAP to 2D...
done.


In [ ]:
# full dataset UMAP — colored by coarse class
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_facecolor('#070810')
fig.patch.set_facecolor('#070810')

for label, color in COARSE_COLORS.items():
    mask = np.array(labels) == label
    if mask.sum() == 0:
        continue
    ax.scatter(embs_2d[mask, 0], embs_2d[mask, 1],
               c=color, s=2, alpha=0.5, label=f'{label} ({mask.sum()})', linewidths=0)

ax.legend(loc='upper right', fontsize=9, framealpha=0.3,
          labelcolor='white', facecolor='#12152a')
ax.set_title('SONYC — CLAP embedding space (UMAP)\neach point = one 10s clip, position = acoustic similarity', color='white', fontsize=13)
ax.set_xlabel('UMAP dimension 1  (no physical meaning — closer = more acoustically similar)', color='#5a5f8a', fontsize=9)
ax.set_ylabel('UMAP dimension 2', color='#5a5f8a', fontsize=9)
ax.tick_params(colors='#5a5f8a')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e2240')
plt.tight_layout()
plt.savefig('plots/umap_coarse.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved plots/umap_coarse.png')

In [ ]:
# UMAP colored by borough (1=Manhattan, 3=Brooklyn, 4=Queens)
BOROUGH_COLORS = {'1': '#74b9ff', '3': '#a29bfe', '4': '#55efc4', '': '#333344'}
BOROUGH_NAMES  = {'1': 'Manhattan', '3': 'Brooklyn', '4': 'Queens', '': 'unknown'}

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_facecolor('#070810')
fig.patch.set_facecolor('#070810')

for b, color in BOROUGH_COLORS.items():
    mask = np.array(boroughs) == b
    if mask.sum() == 0:
        continue
    ax.scatter(embs_2d[mask, 0], embs_2d[mask, 1],
               c=color, s=2, alpha=0.4,
               label=f'{BOROUGH_NAMES[b]} ({mask.sum()})', linewidths=0)

ax.legend(loc='upper right', fontsize=10, framealpha=0.3,
          labelcolor='white', facecolor='#12152a')
ax.set_title('SONYC — same CLAP space, colored by recording borough\noverlapping clusters = boroughs sound alike in those regions', color='white', fontsize=13)
ax.set_xlabel('UMAP dimension 1  (no physical meaning — closer = more acoustically similar)', color='#5a5f8a', fontsize=9)
ax.set_ylabel('UMAP dimension 2', color='#5a5f8a', fontsize=9)
ax.tick_params(colors='#5a5f8a')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e2240')
plt.tight_layout()
plt.savefig('plots/umap_borough.png', dpi=150, bbox_inches='tight')
plt.show()

## Why 93% is "unlabeled"

Not a data quality problem — it's how the SONYC annotation process worked.

The dataset has two annotation tiers:
- **Citizen science** (~18,500 clips): Zooniverse volunteers tagged everything, but with varying agreement and no verification
- **Ground truth** (~1,200 clips, annotator_id=0): SONYC team verified these manually — this is what benchmarks and models use

Our embeddings only attach labels from ground-truth annotations. The other 17k clips are real NYC audio — engines, voices, construction — just not expert-reviewed. CLAP embedded them on acoustics alone, so they land in the right neighborhoods in the UMAP even without a label.

**This is actually useful:** text retrieval (cell 8) can surface unlabeled clips that *sound like* a jackhammer or dog, giving us a much larger pool to pick from than just the 1,200 GT clips.

In [ ]:
# zoomed UMAP — labeled clips bright + large, unlabeled as faint background
labeled_mask = np.array([len(m.get('classes', [])) > 0 for m in meta])
n_labeled = labeled_mask.sum()
print(f'{n_labeled} GT-labeled clips ({n_labeled/len(meta):.1%}) out of {len(meta)}')
print(f'GT means annotator_id=0 (SONYC team verified) — not the same as "has sound"')
print(f'The ~17k unlabeled clips are real city audio, just not expert-reviewed')

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_facecolor('#070810')
fig.patch.set_facecolor('#070810')

# unlabeled as dim background
ax.scatter(embs_2d[~labeled_mask, 0], embs_2d[~labeled_mask, 1],
           c='#12152a', s=1, alpha=0.5, linewidths=0, label=f'unlabeled ({(~labeled_mask).sum()})')

# labeled on top, large + bright
for label, color in COARSE_COLORS.items():
    if label == 'unlabeled':
        continue
    mask = np.array(labels) == label
    if mask.sum() == 0:
        continue
    ax.scatter(embs_2d[mask, 0], embs_2d[mask, 1],
               c=color, s=25, alpha=0.9, label=f'{label} ({mask.sum()})', linewidths=0)

ax.legend(loc='upper right', fontsize=9, framealpha=0.4, labelcolor='white', facecolor='#12152a')
ax.set_title('UMAP — GT-labeled clips highlighted over the full dataset\n'
             'clusters = CLAP learned acoustic similarity without using any labels',
             color='white', fontsize=13)
ax.set_xlabel('UMAP dimension 1  (no physical meaning — closer = sounds more alike)', color='#5a5f8a', fontsize=9)
ax.set_ylabel('UMAP dimension 2', color='#5a5f8a', fontsize=9)
ax.tick_params(colors='#5a5f8a')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e2240')
plt.tight_layout()
plt.savefig('plots/umap_labeled_highlight.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# when do sounds peak? hour × class heatmap
# each row (class) normalized to its own max so rare sounds are still readable

from collections import defaultdict

COARSE_ORDER = ['engine', 'machinery', 'impact', 'saw', 'alert', 'music', 'voice', 'dog']

hour_class = defaultdict(lambda: defaultdict(int))
for m in meta:
    h = m.get('hour', -1)
    if h < 0 or not m.get('classes'):
        continue
    seen = set()
    for cls in m['classes']:
        for fine, coarse in FINE_TO_COARSE.items():
            if fine in cls and coarse not in seen:
                hour_class[h][coarse] += 1
                seen.add(coarse)

matrix = np.array([[hour_class[h].get(c, 0) for c in COARSE_ORDER] for h in range(24)], dtype=float)

# normalize each class column to its own max (so rare classes like 'dog' still show pattern)
col_max = matrix.max(axis=0)
col_max[col_max == 0] = 1
matrix_norm = (matrix / col_max).T  # shape: (n_classes, 24)

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_facecolor('#070810')
fig.patch.set_facecolor('#070810')

im = ax.imshow(matrix_norm, aspect='auto', cmap='magma', interpolation='nearest', vmin=0, vmax=1)
ax.set_yticks(range(len(COARSE_ORDER)))
ax.set_yticklabels(COARSE_ORDER, color='white', fontsize=10)
ax.set_xticks(range(24))
ax.set_xticklabels([f'{h}:00' for h in range(24)], color='#5a5f8a', fontsize=8, rotation=45)
ax.set_title('When do sounds peak? (each row normalized to its own max — shows relative daily rhythm)',
             color='white', fontsize=12)
ax.set_xlabel('hour of day', color='#5a5f8a')

# annotate raw counts (skip zeros)
for j in range(24):
    for i, c in enumerate(COARSE_ORDER):
        v = int(matrix[j, i])
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center', fontsize=6,
                    color='white' if matrix_norm[i, j] > 0.5 else '#666688')

plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label='relative intensity')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e2240')
plt.tight_layout()
plt.savefig('plots/hour_class_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('peak hour per class:')
for i, c in enumerate(COARSE_ORDER):
    peak_h = int(np.argmax(matrix[:, i]))
    peak_count = int(matrix[peak_h, i])
    print(f'  {c:<12} peaks at {peak_h:02d}:00  ({peak_count} clips)')

In [ ]:
# borough × sound class — what share of each borough's GT clips is each sound type?
from collections import defaultdict

COARSE_ORDER = ['engine', 'machinery', 'impact', 'saw', 'alert', 'music', 'voice', 'dog']
BOROUGH_ORDER = ['1', '3', '4']
BOROUGH_LABEL = {'1': 'Manhattan', '3': 'Brooklyn', '4': 'Queens'}

bor_class = defaultdict(lambda: defaultdict(int))
for m in meta:
    b = m.get('borough', '')
    if b not in BOROUGH_ORDER or not m.get('classes'):
        continue
    seen = set()
    for cls in m['classes']:
        for fine, coarse in FINE_TO_COARSE.items():
            if fine in cls and coarse not in seen:
                bor_class[b][coarse] += 1
                seen.add(coarse)

matrix_b = np.array([[bor_class[b].get(c, 0) for c in COARSE_ORDER] for b in BOROUGH_ORDER], dtype=float)
row_sums = matrix_b.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
matrix_b_norm = matrix_b / row_sums

fig, ax = plt.subplots(figsize=(12, 3))
ax.set_facecolor('#070810')
fig.patch.set_facecolor('#070810')

im = ax.imshow(matrix_b_norm, aspect='auto', cmap='magma', interpolation='nearest')
ax.set_yticks(range(3))
ax.set_yticklabels([BOROUGH_LABEL[b] for b in BOROUGH_ORDER], color='white', fontsize=11)
ax.set_xticks(range(len(COARSE_ORDER)))
ax.set_xticklabels(COARSE_ORDER, color='#5a5f8a', fontsize=10)
ax.set_title("Sound class share by borough (% of that borough's GT-labeled clips)", color='white', fontsize=12)

for i, b in enumerate(BOROUGH_ORDER):
    for j, c in enumerate(COARSE_ORDER):
        raw = int(matrix_b[i, j])
        pct = matrix_b_norm[i, j] * 100
        if raw > 0:
            ax.text(j, i, f'{pct:.0f}%\n({raw})', ha='center', va='center',
                    fontsize=8, color='white' if matrix_b_norm[i, j] > 0.3 else '#888888')

plt.colorbar(im, ax=ax, fraction=0.03, pad=0.01)
for spine in ax.spines.values():
    spine.set_edgecolor('#1e2240')
plt.tight_layout()
plt.savefig('plots/borough_class_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# also print totals per borough
for b in BOROUGH_ORDER:
    total = sum(bor_class[b].values())
    print(f'{BOROUGH_LABEL[b]:12s}: {total} GT-labeled sound occurrences')

In [ ]:
# per-class: find 5 most representative (near centroid) + 5 most outlier
# output: candidates.json for Gemini captioning later

FINE_CLASSES = [
    'small-engine', 'medium-engine', 'large-engine',
    'rock-drill', 'jackhammer', 'hoe-ram', 'pile-driver',
    'impact', 'chainsaw', 'small-saw', 'large-saw',
    'car-horn', 'car-alarm', 'siren', 'reverse-beeper',
    'stationary-music', 'mobile-music', 'ice-cream-truck',
    'talking', 'shouting', 'large-crowd', 'amplified-speech', 'dog',
]

# fine class name -> list of indices
FINE_COL_NAMES = {
    'small-engine': 'small-sounding-engine',
    'medium-engine': 'medium-sounding-engine',
    'large-engine': 'large-sounding-engine',
    'small-saw': 'small-medium-rotating-saw',
    'large-saw': 'large-rotating-saw',
    'talking': 'person-or-small-group-talking',
    'shouting': 'person-or-small-group-shouting',
    'large-crowd': 'large-crowd',
    'dog': 'dog-barking-whining',
}

def class_indices(fine_class):
    search = FINE_COL_NAMES.get(fine_class, fine_class)
    return [i for i, m in enumerate(meta) if any(search in c for c in m.get('classes', []))]

candidates = {}
for cls in FINE_CLASSES:
    idxs = class_indices(cls)
    if len(idxs) < 5:
        print(f'  {cls}: only {len(idxs)} clips, skipping')
        continue
    cls_embs = embs[idxs]
    centroid = cls_embs.mean(axis=0)
    centroid = centroid / np.linalg.norm(centroid)
    # cosine similarity to centroid (embs already unit-norm)
    sims = cls_embs @ centroid
    order = np.argsort(sims)  # ascending = outliers first
    top_representative = [idxs[i] for i in order[-5:][::-1]]   # highest sim
    top_outlier        = [idxs[i] for i in order[:5]]           # lowest sim
    candidates[cls] = {
        'n_total': len(idxs),
        'representative': [{'idx': int(i), **meta[i], 'centroid_sim': float(sims[order.tolist().index(idxs.index(i))])} for i in top_representative],
        'outlier':        [{'idx': int(i), **meta[i], 'centroid_sim': float(sims[order.tolist().index(idxs.index(i))])} for i in top_outlier],
    }
    print(f'  {cls:25s} {len(idxs):5d} clips | rep sim: {sims[order[-1]]:.3f} | outlier sim: {sims[order[0]]:.3f}')

with open('outputs/candidates.json', 'w') as f:
    json.dump(candidates, f, indent=2)
print('\nsaved outputs/candidates.json')

In [ ]:
# borough × hour heatmap — how much do Manhattan and Brooklyn differ acoustically?
# compute average embedding per borough per hour, then cosine distance between boroughs

from sklearn.metrics.pairwise import cosine_distances

borough_hour_embs = {}  # (borough, hour) -> mean embedding
for i, m in enumerate(meta):
    b = m.get('borough', '')
    h = m.get('hour', -1)
    if b not in ('1', '3') or h < 0:
        continue
    key = (b, h)
    if key not in borough_hour_embs:
        borough_hour_embs[key] = []
    borough_hour_embs[key].append(embs[i])

# average and normalize
avg = {k: normalize(np.mean(v, axis=0, keepdims=True))[0] for k, v in borough_hour_embs.items()}

# cosine distance between Manhattan and Brooklyn per hour
hours_range = range(24)
distances = []
for h in hours_range:
    m_emb = avg.get(('1', h))
    b_emb = avg.get(('3', h))
    if m_emb is not None and b_emb is not None:
        d = float(cosine_distances([m_emb], [b_emb])[0][0])
    else:
        d = None
    distances.append(d)

# plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.set_facecolor('#070810')
fig.patch.set_facecolor('#070810')

valid_h = [h for h, d in zip(hours_range, distances) if d is not None]
valid_d = [d for d in distances if d is not None]

ax.plot(valid_h, valid_d, color='#a29bfe', linewidth=2)
ax.fill_between(valid_h, valid_d, alpha=0.2, color='#a29bfe')
ax.set_xlabel('hour of day', color='#5a5f8a')
ax.set_ylabel('cosine distance\n(Manhattan vs Brooklyn)', color='#5a5f8a')
ax.set_title('How acoustically different are Manhattan and Brooklyn by hour?', color='white')
ax.set_xticks(range(0, 24, 3))
ax.set_xticklabels([f'{h}:00' for h in range(0, 24, 3)], color='#5a5f8a')
ax.tick_params(colors='#5a5f8a')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e2240')
plt.tight_layout()
plt.savefig('plots/borough_distance_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

peak_h = valid_h[np.argmax(valid_d)]
min_h  = valid_h[np.argmin(valid_d)]
print(f'most different hour:  {peak_h}:00 (distance {max(valid_d):.4f})')
print(f'most similar hour:   {min_h}:00  (distance {min(valid_d):.4f})')

In [ ]:
# CLAP text retrieval — find clips nearest to a text query
# this uses the text encoder of the same model

import torch
from transformers import ClapModel, ClapProcessor

MODEL_ID = 'laion/larger_clap_music_and_speech'
processor = ClapProcessor.from_pretrained(MODEL_ID)
model = ClapModel.from_pretrained(MODEL_ID)
model.eval()

def text_query(query, top_k=10):
    """find top_k clips most similar to the text query"""
    inputs = processor(text=[query], return_tensors='pt', padding=True)
    with torch.no_grad():
        # get_text_features() returns BaseModelOutputWithPooling in transformers 5.x
        # go through text_model + text_projection directly
        text_out = model.text_model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
        )
        text_emb = model.text_projection(text_out.pooler_output)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)
    text_emb_np = text_emb.cpu().float().numpy()[0]
    sims = embs @ text_emb_np
    top_idxs = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_idxs:
        m = meta[idx]
        results.append({
            'sim': round(float(sims[idx]), 4),
            'filename': m['filename'],
            'archive': m['archive'],
            'borough': m.get('borough', ''),
            'hour': m.get('hour', -1),
            'classes': m.get('classes', []),
        })
    return results

# try a few
for q in [
    'jackhammer on a city street',
    'dog barking in an urban park',
    'music playing from a passing car',
    'busy intersection with horns and sirens',
    'quiet street at night',
]:
    results = text_query(q, top_k=5)
    print(f'\n"{q}"')
    for r in results:
        b = {'1': 'Manhattan', '3': 'Brooklyn', '4': 'Queens'}.get(r['borough'], '?')
        print(f'  [{r["sim"]:.3f}] {r["filename"]}  h={r["hour"]}  {b}  {r["classes"][:2]}')

In [ ]:
# your own queries — edit and re-run this cell freely
my_query = 'brooklyn street music at night'

results = text_query(my_query, top_k=15)
print(f'top results for: "{my_query}"\n')
for r in results:
    b = {'1': 'Manhattan', '3': 'Brooklyn', '4': 'Queens'}.get(r['borough'], '?')
    print(f'  [{r["sim"]:.3f}] {r["filename"]}  h={r["hour"]:2d}  {b:<12} {r["classes"][:3]}')

## Gemini caption EDA

18,510 Gemini captions — one per SONYC clip. Each caption is a ~2–4 sentence description of what Gemini heard.
Questions: how long are they, do they use the right words, where does Gemini hedge, what varies by borough/hour?

In [ ]:
import json
import re
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# load all 18k captions
captions = []
with open('outputs/captions.jsonl') as f:
    for line in f:
        captions.append(json.loads(line.strip()))

gt     = [c for c in captions if c.get('gt')]
non_gt = [c for c in captions if not c.get('gt')]

lengths = [len(c['caption']) for c in captions]
words   = [len(c['caption'].split()) for c in captions]

print(f'total captions:  {len(captions):,}')
print(f'  gt-labeled:    {len(gt):,}  ({len(gt)/len(captions):.1%})')
print(f'  non-gt:        {len(non_gt):,}')
print(f'caption length (chars):  mean={np.mean(lengths):.0f}  median={np.median(lengths):.0f}  min={min(lengths)}  max={max(lengths)}')
print(f'caption length (words):  mean={np.mean(words):.0f}  median={np.median(words):.0f}')

# length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor('#070810')
for ax in axes:
    ax.set_facecolor('#070810')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e2240')
    ax.tick_params(colors='#5a5f8a')

axes[0].hist(lengths, bins=60, color='#a29bfe', alpha=0.8, edgecolor='none')
axes[0].axvline(np.mean(lengths), color='#fd79a8', linewidth=1.5, label=f'mean {np.mean(lengths):.0f}')
axes[0].axvline(np.median(lengths), color='#74b9ff', linewidth=1.5, linestyle='--', label=f'median {np.median(lengths):.0f}')
axes[0].set_title('caption length (chars)', color='white')
axes[0].set_xlabel('characters', color='#5a5f8a')
axes[0].legend(fontsize=9, labelcolor='white', framealpha=0.3, facecolor='#12152a')

axes[1].hist(words, bins=40, color='#55efc4', alpha=0.8, edgecolor='none')
axes[1].axvline(np.mean(words), color='#fd79a8', linewidth=1.5, label=f'mean {np.mean(words):.0f}')
axes[1].set_title('caption length (words)', color='white')
axes[1].set_xlabel('words', color='#5a5f8a')
axes[1].legend(fontsize=9, labelcolor='white', framealpha=0.3, facecolor='#12152a')

plt.suptitle('Gemini caption length distribution  (n=18,510)', color='white', fontsize=12)
plt.tight_layout()
plt.savefig('plots/caption_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# keyword hit rate per fine class — does Gemini use the right words?
# mirrors analyze_captions.py KEYWORDS but visualized

KEYWORDS = {
    'small-sounding-engine':         ['scooter', 'motorcycle', 'moped', 'small engine', 'small vehicle', 'motorbike'],
    'medium-sounding-engine':        ['medium engine', 'car engine', 'sedan', 'passenger car', 'medium vehicle', 'car'],
    'large-sounding-engine':         ['large engine', 'bus', 'truck', 'heavy vehicle', 'large vehicle', 'diesel'],
    'rock-drill':                    ['rock drill', 'drilling', 'drill', 'boring machine', 'construction machinery', 'construction'],
    'jackhammer':                    ['jackhammer', 'jack hammer', 'pneumatic', 'impact wrench', 'hammering', 'pounding', 'percussive'],
    'hoe-ram':                       ['hoe-ram', 'hoe ram', 'excavator', 'hydraulic hammer', 'demolition'],
    'pile-driver':                   ['pile driver', 'piledriver', 'pile driving'],
    'non-machinery-impact':          ['bang', 'knock', 'thud', 'crash', 'slam', 'impact', 'clatter', 'metallic clang', 'clanking', 'clanging'],
    'chainsaw':                      ['chainsaw', 'chain saw', 'weed trimmer', 'power tool'],
    'small-medium-rotating-saw':     ['circular saw', 'rotating saw', 'angle grinder', 'saw', 'metallic grinding', 'high-pitched whine', 'whirring'],
    'large-rotating-saw':            ['large saw', 'large rotating saw', 'industrial saw', 'metallic grinding', 'grinding'],
    'car-horn':                      ['car horn', 'horn', 'honk', 'honking'],
    'car-alarm':                     ['car alarm', 'alarm'],
    'siren':                         ['siren', 'emergency vehicle', 'ambulance', 'police siren', 'fire truck', 'wail', 'wailing'],
    'reverse-beeper':                ['reverse beeper', 'reversing', 'backing up', 'beeping', 'beep', 'reversing alarm'],
    'stationary-music':              ['music', 'busker', 'playing music', 'musical', 'song'],
    'mobile-music':                  ['mobile music', 'passing music', 'car stereo', 'music from a vehicle', 'music from a passing', 'musical track'],
    'ice-cream-truck':               ['ice cream truck', 'ice-cream truck', 'ice cream', 'jingle', 'chime', 'melodic'],
    'person-or-small-group-talking': ['talking', 'conversation', 'speech', 'speaking', 'voices', 'chatter'],
    'person-or-small-group-shouting':['shouting', 'yelling', 'calling out', 'shout', 'exclamation'],
    'large-crowd':                   ['crowd', 'large crowd', 'many voices', 'cheering', 'crowd noise', 'group of people'],
    'amplified-speech':              ['amplified', 'megaphone', 'loudspeaker', 'announcement', 'public address', 'pa system'],
    'dog-barking-whining':           ['dog', 'bark', 'barking', 'whining', 'canine', 'yelp'],
}

# index gt captions by fine class
gt_by_class = defaultdict(list)
for c in captions:
    if not c.get('gt'):
        continue
    for cls in c.get('classes', []):
        if cls in KEYWORDS:
            gt_by_class[cls].append(c)

# for each class: what % of captions hit at least one keyword?
# also: which keyword is hit most often?
stats = {}
for cls, kws in KEYWORDS.items():
    pool = gt_by_class[cls]
    if not pool:
        stats[cls] = {'n': 0, 'hit_rate': 0, 'top_kw': '', 'top_kw_rate': 0, 'per_kw': {}}
        continue
    hit = sum(1 for c in pool if any(kw in c['caption'].lower() for kw in kws))
    per_kw = {kw: sum(1 for c in pool if kw in c['caption'].lower()) / len(pool) for kw in kws}
    top_kw = max(per_kw, key=per_kw.get)
    stats[cls] = {
        'n': len(pool),
        'hit_rate': hit / len(pool),
        'top_kw': top_kw,
        'top_kw_rate': per_kw[top_kw],
        'per_kw': per_kw,
    }

# sort by hit rate
order = sorted(KEYWORDS.keys(), key=lambda c: stats[c]['hit_rate'])
hit_rates = [stats[c]['hit_rate'] for c in order]
ns        = [stats[c]['n'] for c in order]

fig, ax = plt.subplots(figsize=(10, 9))
fig.patch.set_facecolor('#070810')
ax.set_facecolor('#070810')

colors_bar = ['#26de81' if r >= 0.6 else '#a29bfe' if r >= 0.3 else '#fd79a8' for r in hit_rates]
bars = ax.barh(order, hit_rates, color=colors_bar, alpha=0.85, height=0.65)

# annotate with n and top keyword
for i, (cls, bar) in enumerate(zip(order, bars)):
    s = stats[cls]
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'n={s["n"]}  "{s["top_kw"]}" ({s["top_kw_rate"]:.0%})',
            va='center', fontsize=7.5, color='#9090c0')

ax.axvline(0.5, color='#5a5f8a', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlim(0, 1.4)
ax.set_xlabel('fraction of GT clips where caption hits any keyword', color='#5a5f8a')
ax.set_title('Gemini keyword hit rate per fine class\ngreen ≥60% | purple 30–60% | pink <30%', color='white', fontsize=12)
ax.tick_params(colors='#9090c0', labelsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#1e2240')

plt.tight_layout()
plt.savefig('plots/caption_kw_hit_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nclasses with low keyword coverage (< 40%) — Gemini uses different language:')
for cls in order:
    if stats[cls]['hit_rate'] < 0.4 and stats[cls]['n'] > 0:
        print(f'  {cls:<35}  {stats[cls]["hit_rate"]:.0%}  (n={stats[cls]["n"]})')

In [ ]:
# what words does Gemini actually use? top N-grams across all captions
# useful for understanding its vocabulary vs our keyword lists

import re
from collections import Counter

STOP = {
    'the','a','an','of','in','to','and','is','are','this','that','with','it',
    'as','by','on','from','its','be','at','for','or','which','there','these',
    'their','into','has','have','been','within','can','was','also','throughout',
    'creating','suggesting','indicating','overall','very','more','most','some',
    'other','what','while','where','when','such','both','each','during','through',
    'out','up','often','quite','somewhat','relatively','slightly',
}

def tokenize(text):
    return [w for w in re.findall(r"[a-z'-]+", text.lower()) if w not in STOP and len(w) > 2]

def ngrams(tokens, n):
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

all_tokens = [tok for c in captions for tok in tokenize(c['caption'])]
uni  = Counter(all_tokens)
bi   = Counter(gram for c in captions for gram in ngrams(tokenize(c['caption']), 2))
tri  = Counter(gram for c in captions for gram in ngrams(tokenize(c['caption']), 3))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#070810')

palette = ['#a29bfe', '#74b9ff', '#55efc4']
titles  = ['top 25 unigrams', 'top 25 bigrams', 'top 25 trigrams']
counters = [uni, bi, tri]

for ax, ctr, title, color in zip(axes, counters, titles, palette):
    ax.set_facecolor('#070810')
    top = ctr.most_common(25)
    terms, counts = zip(*top)
    ax.barh(list(reversed(terms)), list(reversed(counts)), color=color, alpha=0.8, height=0.7)
    ax.set_title(title, color='white', fontsize=11)
    ax.tick_params(colors='#9090c0', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e2240')

plt.suptitle("Gemini's vocabulary across all 18,510 captions", color='white', fontsize=13)
plt.tight_layout()
plt.savefig('plots/caption_ngrams.png', dpi=150, bbox_inches='tight')
plt.show()

# also: what % of captions start with "This 10-second"?
boilerplate = sum(1 for c in captions if c['caption'].startswith('This 10-second'))
print(f'"This 10-second..." opener: {boilerplate:,} / {len(captions):,} = {boilerplate/len(captions):.1%}')
other_openers = Counter(c['caption'].split('.')[0][:30] for c in captions if not c['caption'].startswith('This 10-second'))
print('\nother common openers:')
for opener, n in other_openers.most_common(10):
    print(f'  ({n:4d}x)  {opener}...')

In [ ]:
# hedging language — how often does Gemini express uncertainty?
# split by gt vs non-gt and by coarse class

HEDGE_WORDS = ['likely', 'possibly', 'perhaps', 'appears', 'suggesting', 'suggests',
               'probably', 'might', 'could', 'seem', 'seems', 'consistent with',
               'indicative of', 'what sounds like', 'may be', 'resembles']

COARSE_MAP = {
    'small-sounding-engine': 'engine', 'medium-sounding-engine': 'engine', 'large-sounding-engine': 'engine',
    'rock-drill': 'machinery', 'jackhammer': 'machinery', 'hoe-ram': 'machinery', 'pile-driver': 'machinery',
    'non-machinery-impact': 'impact',
    'chainsaw': 'saw', 'small-medium-rotating-saw': 'saw', 'large-rotating-saw': 'saw',
    'car-horn': 'alert', 'car-alarm': 'alert', 'siren': 'alert', 'reverse-beeper': 'alert',
    'stationary-music': 'music', 'mobile-music': 'music', 'ice-cream-truck': 'music',
    'person-or-small-group-talking': 'voice', 'person-or-small-group-shouting': 'voice',
    'large-crowd': 'voice', 'amplified-speech': 'voice',
    'dog-barking-whining': 'dog',
}
COARSE_ORDER = ['engine', 'machinery', 'impact', 'saw', 'alert', 'music', 'voice', 'dog']
COARSE_COLORS = {
    'engine': '#ff6b6b', 'machinery': '#ff9f43', 'impact': '#ffd32a',
    'saw': '#26de81', 'alert': '#fd79a8', 'music': '#a29bfe',
    'voice': '#74b9ff', 'dog': '#55efc4',
}

def hedge_count(text):
    t = text.lower()
    return sum(1 for h in HEDGE_WORDS if h in t)

def has_hedge(text):
    return hedge_count(text) > 0

# overall hedge rates
gt_hedge   = sum(1 for c in gt     if has_hedge(c['caption'])) / len(gt)
ngt_hedge  = sum(1 for c in non_gt if has_hedge(c['caption'])) / len(non_gt)
print(f'hedge rate — gt: {gt_hedge:.1%}   non-gt: {ngt_hedge:.1%}')

# per coarse class hedge rate (gt only)
coarse_hedge = {}
for coarse in COARSE_ORDER:
    fine_classes = [f for f, c in COARSE_MAP.items() if c == coarse]
    pool = [cap for cap in captions if cap.get('gt') and any(f in cap.get('classes', []) for f in fine_classes)]
    if pool:
        coarse_hedge[coarse] = sum(1 for c in pool if has_hedge(c['caption'])) / len(pool)
    else:
        coarse_hedge[coarse] = 0

# per coarse class: mean hedge count
coarse_hedge_count = {}
for coarse in COARSE_ORDER:
    fine_classes = [f for f, c in COARSE_MAP.items() if c == coarse]
    pool = [cap for cap in captions if cap.get('gt') and any(f in cap.get('classes', []) for f in fine_classes)]
    coarse_hedge_count[coarse] = np.mean([hedge_count(c['caption']) for c in pool]) if pool else 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#070810')

# left: bar per class — % captions with any hedge
ax = axes[0]
ax.set_facecolor('#070810')
vals  = [coarse_hedge[c] for c in COARSE_ORDER]
cols  = [COARSE_COLORS[c] for c in COARSE_ORDER]
ax.bar(COARSE_ORDER, vals, color=cols, alpha=0.85, width=0.6)
ax.axhline(gt_hedge, color='white', linewidth=1, linestyle='--', alpha=0.5, label=f'overall gt avg {gt_hedge:.0%}')
ax.set_ylabel('fraction with ≥1 hedge word', color='#5a5f8a')
ax.set_title('Gemini hedging rate by sound class\n(gt clips only)', color='white')
ax.tick_params(colors='#9090c0', labelsize=9)
ax.set_ylim(0, 1)
ax.legend(fontsize=8, labelcolor='white', framealpha=0.3, facecolor='#12152a')
for spine in ax.spines.values(): spine.set_edgecolor('#1e2240')

# right: mean hedge word count per class
ax = axes[1]
ax.set_facecolor('#070810')
vals2 = [coarse_hedge_count[c] for c in COARSE_ORDER]
ax.bar(COARSE_ORDER, vals2, color=cols, alpha=0.85, width=0.6)
ax.set_ylabel('mean hedge word count per caption', color='#5a5f8a')
ax.set_title('Hedge intensity — more = Gemini less confident', color='white')
ax.tick_params(colors='#9090c0', labelsize=9)
for spine in ax.spines.values(): spine.set_edgecolor('#1e2240')

plt.suptitle('Gemini uncertainty / hedging language', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('plots/caption_hedging.png', dpi=150, bbox_inches='tight')
plt.show()

# per-hedge-word breakdown
print('\nwhich hedge words appear most?')
for hw in sorted(HEDGE_WORDS, key=lambda w: -sum(1 for c in captions if w in c['caption'].lower())):
    n = sum(1 for c in captions if hw in c['caption'].lower())
    print(f'  {hw:<25} {n:6,}  ({n/len(captions):.1%})')

In [ ]:
# caption richness by hour and borough
# do descriptions get richer (longer, more specific) at certain times of day?

hour_lengths  = defaultdict(list)
hour_hedges   = defaultdict(list)
borough_lengths = defaultdict(list)

for c in captions:
    h = c.get('hour', -1)
    b = c.get('borough', '')
    cl = len(c['caption'])
    hg = hedge_count(c['caption'])
    if 0 <= h <= 23:
        hour_lengths[h].append(cl)
        hour_hedges[h].append(hg)
    if b in ('1', '3', '4'):
        borough_lengths[b].append(cl)

hrs = list(range(24))
mean_len   = [np.mean(hour_lengths[h]) if hour_lengths[h] else 0 for h in hrs]
mean_hedge = [np.mean(hour_hedges[h])  if hour_hedges[h]  else 0 for h in hrs]
n_per_hour = [len(hour_lengths[h]) for h in hrs]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.patch.set_facecolor('#070810')

for ax in axes:
    ax.set_facecolor('#070810')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e2240')
    ax.tick_params(colors='#5a5f8a')

axes[0].plot(hrs, mean_len, color='#a29bfe', linewidth=2)
axes[0].fill_between(hrs, mean_len, alpha=0.15, color='#a29bfe')
axes[0].set_ylabel('mean caption length\n(chars)', color='#5a5f8a', fontsize=9)
axes[0].set_title('Caption richness, hedging, and clip count by hour of day', color='white', fontsize=12)

axes[1].plot(hrs, mean_hedge, color='#fd79a8', linewidth=2)
axes[1].fill_between(hrs, mean_hedge, alpha=0.15, color='#fd79a8')
axes[1].set_ylabel('mean hedge words\nper caption', color='#5a5f8a', fontsize=9)

axes[2].bar(hrs, n_per_hour, color='#74b9ff', alpha=0.7, width=0.8)
axes[2].set_ylabel('# clips', color='#5a5f8a', fontsize=9)
axes[2].set_xlabel('hour of day', color='#5a5f8a')
axes[2].set_xticks(hrs)
axes[2].set_xticklabels([f'{h}' for h in hrs], fontsize=8)

plt.tight_layout()
plt.savefig('plots/caption_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

# borough comparison
BNAMES = {'1': 'Manhattan', '3': 'Brooklyn', '4': 'Queens'}
print('mean caption length by borough:')
for b in ['1', '3', '4']:
    lens = borough_lengths[b]
    print(f'  {BNAMES[b]:12s}  n={len(lens):6,}  mean={np.mean(lens):.0f}  median={np.median(lens):.0f}')

In [ ]:
# class-distinctive words — what words appear much more in one class vs. all others?
# simple TF-IDF-style: for each coarse class, find words with highest ratio
# class_freq / corpus_freq

corpus_freq = Counter()
class_tokens = defaultdict(list)

for c in captions:
    toks = tokenize(c['caption'])
    corpus_freq.update(toks)
    # assign to coarse class if gt
    if c.get('gt') and c.get('classes'):
        coarse_set = set()
        for fine in c['classes']:
            if fine in COARSE_MAP:
                coarse_set.add(COARSE_MAP[fine])
        for coarse in coarse_set:
            class_tokens[coarse].extend(toks)

total_tokens = sum(corpus_freq.values())

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.patch.set_facecolor('#070810')

for ax, coarse in zip(axes.flat, COARSE_ORDER):
    ax.set_facecolor('#070810')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e2240')
    ax.tick_params(colors='#9090c0', labelsize=8)

    toks = class_tokens[coarse]
    if not toks:
        ax.set_title(coarse, color='white')
        continue
    class_freq = Counter(toks)
    class_total = sum(class_freq.values())

    # ratio: (class_count / class_total) / (corpus_count / total_tokens)
    # only words that appear ≥5x in class
    ratios = {}
    for word, cnt in class_freq.items():
        if cnt < 5 or len(word) < 3:
            continue
        class_rate  = cnt / class_total
        corpus_rate = corpus_freq[word] / total_tokens
        ratios[word] = class_rate / corpus_rate

    top = sorted(ratios, key=ratios.get, reverse=True)[:15]
    vals = [ratios[w] for w in top]

    color = COARSE_COLORS[coarse]
    ax.barh(list(reversed(top)), list(reversed(vals)), color=color, alpha=0.8, height=0.7)
    ax.set_title(coarse, color=color, fontsize=11, fontweight='bold')
    ax.set_xlabel('relative frequency vs. corpus', color='#5a5f8a', fontsize=8)

plt.suptitle('Class-distinctive words in Gemini captions\n(words most over-represented in each sound class vs. all captions)',
             color='white', fontsize=12)
plt.tight_layout()
plt.savefig('plots/caption_class_words.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# sample best and worst captions — show actual text for manual review
# "best" = keyword match + short (specific, not padded)
# "worst" = no keyword match, long and generic-sounding

from textwrap import fill

def kw_hit(c, cls):
    return any(kw in c['caption'].lower() for kw in KEYWORDS.get(cls, []))

print('=' * 80)
print('BEST CAPTIONS (gt, keyword match, concise)')
print('=' * 80)

for cls in ['dog-barking-whining', 'siren', 'jackhammer', 'stationary-music', 'large-crowd']:
    pool = [c for c in captions if c.get('gt') and cls in c.get('classes', []) and kw_hit(c, cls)]
    if not pool:
        continue
    # shortest that still hits keyword = concise + accurate
    best = sorted(pool, key=lambda c: len(c['caption']))[:1][0]
    print(f'\n[{cls}]  ({len(best["caption"])} chars, h={best["hour"]}, {BNAMES.get(best["borough"], "?")})')
    print(fill(best['caption'], width=80, initial_indent='  ', subsequent_indent='  '))

print()
print('=' * 80)
print('WORST CAPTIONS (gt, no keyword match — Gemini missed the sound)')
print('=' * 80)

for cls in ['jackhammer', 'hoe-ram', 'mobile-music', 'reverse-beeper', 'pile-driver']:
    pool = [c for c in captions if c.get('gt') and cls in c.get('classes', []) and not kw_hit(c, cls)]
    if not pool:
        print(f'\n[{cls}]  all captions hit keyword — good coverage')
        continue
    # longest no-hit = most likely to be generic padding
    worst = sorted(pool, key=lambda c: -len(c['caption']))[:1][0]
    print(f'\n[{cls}]  ({len(worst["caption"])} chars, h={worst["hour"]}, {BNAMES.get(worst["borough"], "?")})')
    print(fill(worst['caption'], width=80, initial_indent='  ', subsequent_indent='  '))